In [19]:
import numpy as np
import pandas as pd
import joblib
import os
from PIL import Image

import matplotlib.pyplot as plt
import seaborn as sns

from scipy.stats import randint, uniform, norm, loguniform

from sklearn.preprocessing import StandardScaler

from sklearn.metrics import classification_report

from sklearn.pipeline import Pipeline

from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, ExtraTreesClassifier, HistGradientBoostingClassifier
from sklearn.svm import SVC, LinearSVC
from sklearn.linear_model import LogisticRegression, SGDClassifier

from sklearn.model_selection import StratifiedKFold, RandomizedSearchCV, GridSearchCV

from config import SEED, N_MELS

Elegimos dataset

In [20]:
dataset = 'bp_wd2'

## Espectrogramas

Se suelen usar Mel espectrogramas

In [4]:
train_data = np.load(f'./dataset/ciclos_procesados/{dataset}/train_melspectrogram.npz')
X_train = train_data['X']
y_train = train_data['y']

test_data = np.load(f'./dataset/ciclos_procesados/{dataset}/test_melspectrogram.npz')
X_test = test_data['X']
y_test = test_data['y']

In [5]:
X_train.shape, y_train.shape, X_test.shape, y_test.shape

((13469, 47616), (13469,), (3239, 47616), (3239,))

In [6]:
np.random.seed(SEED)

i_samples = np.random.choice(X_train.shape[0], size=5000, replace=False)

### Random Forest

#### Entrenamiento

In [7]:
rf = RandomForestClassifier(random_state=SEED, n_jobs=-1, max_features='sqrt')
param_distributions = {
    'max_depth': randint(25, 50),
    'min_samples_split': [30, 40, 50],
    'min_samples_leaf': [15, 20, 25],
}

In [12]:
rnd_search = RandomizedSearchCV(
    estimator=rf,
    param_distributions=param_distributions,
    n_iter=10,
    scoring='roc_auc',
    cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED),
    verbose=3,
    random_state=SEED
)

rnd_search.fit(X_train, y_train)

Fitting 5 folds for each of 10 candidates, totalling 50 fits
[CV 1/5] END max_depth=31, min_samples_leaf=15, min_samples_split=50;, score=0.663 total time= 1.4min
[CV 2/5] END max_depth=31, min_samples_leaf=15, min_samples_split=50;, score=0.661 total time= 1.0min
[CV 3/5] END max_depth=31, min_samples_leaf=15, min_samples_split=50;, score=0.662 total time= 1.0min
[CV 4/5] END max_depth=31, min_samples_leaf=15, min_samples_split=50;, score=0.657 total time= 1.0min
[CV 5/5] END max_depth=31, min_samples_leaf=15, min_samples_split=50;, score=0.652 total time= 1.1min
[CV 1/5] END max_depth=35, min_samples_leaf=15, min_samples_split=30;, score=0.659 total time=  60.0s
[CV 2/5] END max_depth=35, min_samples_leaf=15, min_samples_split=30;, score=0.667 total time= 1.0min
[CV 3/5] END max_depth=35, min_samples_leaf=15, min_samples_split=30;, score=0.662 total time= 1.0min
[CV 4/5] END max_depth=35, min_samples_leaf=15, min_samples_split=30;, score=0.661 total time= 1.0min
[CV 5/5] END max_dept

KeyboardInterrupt: 

In [8]:
pd.DataFrame(rnd_search.cv_results_)[['param_max_depth', 'param_min_samples_leaf', 'param_min_samples_split', 'mean_test_score']].sort_values(by='mean_test_score', ascending=False).head(10)

,param_max_depth,param_min_samples_leaf,param_min_samples_split,mean_test_score
9,20,20,15,0.662697
5,15,10,25,0.662349
3,20,15,15,0.661290
2,15,10,15,0.659032
7,15,20,25,0.657831
6,15,20,20,0.657831
8,15,15,15,0.657111
1,15,15,20,0.657111
4,10,10,15,0.654487
0,10,20,25,0.651882


In [ ]:
print("Best params:", rnd_search.best_params_)
print("Best CV score:", rnd_search.best_score_)

best_model = rnd_search.best_estimator_

In [8]:
best_model = RandomForestClassifier(
    max_depth=15,
    min_samples_leaf=10,
    min_samples_split=15,
    random_state=SEED,
    n_jobs=-1,
    max_features='sqrt'
)

In [9]:
best_model.fit(X_train, y_train)

,n_estimators,100
,criterion,'gini'
,max_depth,15
,min_samples_split,15
,min_samples_leaf,10
,min_weight_fraction_leaf,0.0
,max_features,'sqrt'
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,bootstrap,True
,oob_score,False


In [10]:
y_pred = best_model.predict(X_train)
print(classification_report(y_train, y_pred))

              precision    recall  f1-score   support

           0       0.97      0.96      0.96      6907
           1       0.96      0.96      0.96      6562

    accuracy                           0.96     13469
   macro avg       0.96      0.96      0.96     13469
weighted avg       0.96      0.96      0.96     13469



#### Evaluación

In [11]:
y_pred = best_model.predict(X_test)
print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

           0       0.62      0.63      0.62      1657
           1       0.60      0.60      0.60      1582

    accuracy                           0.61      3239
   macro avg       0.61      0.61      0.61      3239
weighted avg       0.61      0.61      0.61      3239



#### Guardado

In [12]:
os.makedirs(f'./modelos_clasicos/modelos/{dataset}', exist_ok=True)

joblib.dump(best_model, f'./modelos_clasicos/modelos/{dataset}/melspec_rf.pkl')

['./modelos_clasicos/modelos/bp_wd2/melspec_rf.pkl']

## Features de Audio

In [21]:
train_data = np.load(f'./dataset/ciclos_procesados/{dataset}/train_features.npz')
X_train = train_data['X']
y_train = train_data['y']

test_data = np.load(f'./dataset/ciclos_procesados/{dataset}/test_features.npz')
X_test = test_data['X']
y_test = test_data['y']

In [22]:
X_train.shape, y_train.shape, X_test.shape, y_test.shape

((13469, 46), (13469,), (3239, 46), (3239,))

### Random Forest

#### Entrenamiento

In [28]:
rf = RandomForestClassifier(random_state=42, n_jobs=-1, max_features=None)

param_distributions = {
    'max_depth': [10, 15, 20],
    'min_samples_split': [15, 20, 25],
    'min_samples_leaf': [5, 10, 15]
}

In [29]:
rnd_search = RandomizedSearchCV(
    estimator=rf,
    param_distributions=param_distributions,
    n_iter=20,
    scoring='roc_auc',
    cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED),
    verbose=3,
    random_state=SEED
)

rnd_search.fit(X_train, y_train)

Fitting 5 folds for each of 20 candidates, totalling 100 fits
[CV 1/5] END max_depth=10, min_samples_leaf=15, min_samples_split=25;, score=0.727 total time=   5.5s
[CV 2/5] END max_depth=10, min_samples_leaf=15, min_samples_split=25;, score=0.744 total time=   5.4s
[CV 3/5] END max_depth=10, min_samples_leaf=15, min_samples_split=25;, score=0.722 total time=   8.8s
[CV 4/5] END max_depth=10, min_samples_leaf=15, min_samples_split=25;, score=0.744 total time=   6.7s
[CV 5/5] END max_depth=10, min_samples_leaf=15, min_samples_split=25;, score=0.754 total time=   7.2s
[CV 1/5] END max_depth=15, min_samples_leaf=10, min_samples_split=20;, score=0.749 total time=   8.8s
[CV 2/5] END max_depth=15, min_samples_leaf=10, min_samples_split=20;, score=0.759 total time=   8.2s
[CV 3/5] END max_depth=15, min_samples_leaf=10, min_samples_split=20;, score=0.739 total time=   8.1s
[CV 4/5] END max_depth=15, min_samples_leaf=10, min_samples_split=20;, score=0.766 total time=   8.5s
[CV 5/5] END max_dep

,estimator,RandomForestC...ndom_state=42)
,param_distributions,"{'max_depth': [10, 15, ...], 'min_samples_leaf': [5, 10, ...], 'min_samples_split': [15, 20, ...]}"
,n_iter,20
,scoring,'roc_auc'
,n_jobs,None
,refit,True
,cv,StratifiedKFo... shuffle=True)
,verbose,3
,pre_dispatch,'2*n_jobs'
,random_state,42
,error_score,nan


In [30]:
pd.DataFrame(rnd_search.cv_results_)[['param_max_depth', 'param_min_samples_leaf', 'param_min_samples_split', 'mean_test_score']].sort_values(by='mean_test_score', ascending=False).head(10)

,param_max_depth,param_min_samples_leaf,param_min_samples_split,mean_test_score
19,20,5,15,0.763233
2,15,5,15,0.760352
15,20,10,20,0.757846
3,20,10,15,0.757846
8,15,10,15,0.756338
1,15,10,20,0.756338
5,15,5,25,0.756290
18,20,10,25,0.755792
9,20,15,15,0.751301
17,20,15,20,0.751301


In [31]:
print("Best params:", rnd_search.best_params_)
print("Best CV score:", rnd_search.best_score_)

best_model = rnd_search.best_estimator_

Best params: {'min_samples_split': 15, 'min_samples_leaf': 5, 'max_depth': 20}
Best CV score: 0.7632332637227396


In [32]:
y_pred = best_model.predict(X_train)
print(classification_report(y_train, y_pred))

              precision    recall  f1-score   support

           0       0.98      0.98      0.98      6886
           1       0.98      0.98      0.98      6583

    accuracy                           0.98     13469
   macro avg       0.98      0.98      0.98     13469
weighted avg       0.98      0.98      0.98     13469



#### Evaluación

In [33]:
y_pred = best_model.predict(X_test)
print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

           0       0.69      0.68      0.68      1673
           1       0.66      0.68      0.67      1566

    accuracy                           0.68      3239
   macro avg       0.68      0.68      0.68      3239
weighted avg       0.68      0.68      0.68      3239



#### Guardado

In [34]:
joblib.dump(best_model, f'./modelos_clasicos/modelos/{dataset}/features_rf.pkl')

['./modelos_clasicos/modelos/bp_wd2/features_rf.pkl']